In [ ]:
from ruamel import yaml

# maximum distance 
max_distance_double_spot = 1.0

# load YAML analysis "manifest", i.e. list of samples to include 
# plus some metadata to clearly label experimental conditions, e.g. cell type
with open('myc-evi_analysis_manifest_gabi-samples.yaml') as fd:
    YAML = yaml.YAML()
    res = YAML.load(fd)
res

In [ ]:
from pathlib import Path
import pandas as pd

def load_distance_file_add_sample_info(distance_file, spot_file, sample_info):
    df_i = pd.read_csv(distance_file)

    df_spots = pd.read_csv(spot_file)

    # merge spot distances with spot detection (i.e. coordinates)
    # NOTE: we drop the duplicate channel column, otherwise it would show up twice

    # df_i = df_i.merge(df_spots.drop(columns='channel'), left_on='spot_idx', right_index=True)

    df_i = df_i.merge(df_spots.set_index(['channel', 'spot_idx']), left_on=['channel', 'spot_idx'], right_index=True)
    
    # discard channels not in sample info
    df_i = df_i[df_i.channel.isin(sample_info['targets'].keys())]
    
    # add experimental metadata
    df_i['sample'] = sample_info['sample_name']
    df_i['batch'] = sample_info['batch_name']
    df_i['cell_type'] = sample_info['cell_type']
    df_i['target'] = df_i.channel.apply(lambda c: sample_info['targets'][c])
    
    return df_i

def load_all_distance_files_for_sample(sample_info):
    pattern = sample_info["subsample_pattern"] if sample_info["subsample_pattern"] is not None else ""
    distance_files = sorted((Path(sample_info['base_path']) / sample_info['distances_path']).glob(f'{pattern}*.csv'))
    spot_files = sorted((Path(sample_info['base_path']) / sample_info['spot_path']).glob(f'{pattern}*.csv'))
    df = pd.concat(load_distance_file_add_sample_info(distance_file, spot_file, sample_info) for spot_file, distance_file in zip(spot_files, distance_files)).reset_index(drop=True)
    return df

# concatenate all distance files
df = pd.concat(load_all_distance_files_for_sample(sample_info) for sample_info in res).reset_index(drop=True)
df

In [ ]:
df.groupby(['target', 'cell_type']).d.agg('count')

In [ ]:
df.groupby(['target', 'cell_type', 'batch']).d.agg('count')

In [ ]:
import numpy as np

# get double spot yes/no column
# TODO: do this cleanly

split_dfs = []
for _, dfi in df.groupby(['mask_file', 'label']):

    if len(dfi.target.unique()) != 2:
        dfi['double_spot'] = False

    else:
        target_1, target_2 = dfi.target.unique()

        coords_target_1 = dfi[dfi.target == target_1][['z_micron', 'y_micron', 'x_micron']].values
        coords_target_2 = dfi[dfi.target == target_2][['z_micron', 'y_micron', 'x_micron']].values

        def is_close(query_row):
            # get coordinates in other target
            coords = coords_target_1 if query_row.target == target_2 else coords_target_2
            d = coords - query_row[['z_micron', 'y_micron', 'x_micron']].values    

            return np.any(np.linalg.norm(d.astype(float), axis=1) < max_distance_double_spot)

        dfi['double_spot'] = dfi.apply(is_close, axis=1)
    split_dfs.append(dfi)

df = pd.concat(split_dfs)
# df.double_spot.describe()

In [ ]:
### NOTE: no longer needed, we have batch name in analysis manifest

# is sample from first batch/replicate (66-69) or second (72-75) or third (no GS in sample name)
def get_batch_id(sample):
    if 'GS' in sample:
        return '1' if int(sample[-2:]) < 70 else '2'
    else: # AM08-05/-06
        return '3'

# for Gabi samples only
# df['batch'] = np.where(df['sample'].str[-2:].astype(int) > 70, '2', '1')

# df['batch'] = df['sample'].apply(get_batch_id)

In [ ]:
# get only spots from wt evi-myc comparisions or those that are double spots
evi_myc_clone8_samples = ['GS066', 'GS072', '23AM08-05_1']
# df_only_ds = df[df['sample'].isin(evi_myc_clone8_samples) | (~df['sample'].isin(evi_myc_clone8_samples) & df.double_spot)]
df_only_ds = df[(~df['sample'].isin(evi_myc_clone8_samples) & df.double_spot)]

# check sizes (how many spots remain after filter)
df_only_ds.groupby(['sample', 'target']).size()

In [ ]:
import pingouin as pg
import seaborn as sns
from matplotlib import pyplot as plt

sns.set()

df_for_plot = df
# df_for_plot = df_only_ds

dv='d' # raw distance from border
# dv='q' # quantile -> more normalized to cell size?

fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(ax=ax, data=df_for_plot, hue='cell_type', x='target', y=dv, notch=True)
if dv=='d':
    ax.set_ylim((0, 3))

# parwise wilcoxon
pg.pairwise_tests(data=df_for_plot, dv=dv, between=['target', 'cell_type'], parametric=False, padjust='holm')

In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(12,5))

for ax, (t, dfi) in zip(axs, df_for_plot.groupby('target')):
    sns.boxplot(ax=ax, data=dfi, hue='cell_type', x='batch', y=dv, notch=False)
    if dv=='d':
        ax.set_ylim((0, 3))
    ax.set_title(t)
    ax.tick_params(axis='x', rotation=90)

In [ ]:
sns.histplot(data=df, x='gauss_fit_height', hue='batch')
plt.xlim(0,5e3)

df.groupby('batch').gauss_fit_height.describe()